<a href="https://colab.research.google.com/github/Quasardae/Quasardae/blob/main/MCMC_Code_Thesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install emcee corner -q
%matplotlib inline

import os, numpy as np, pandas as pd, emcee, corner
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from scipy.integrate import cumulative_trapezoid, solve_ivp
from scipy.interpolate import interp1d
from scipy.stats import chi2, norm
import urllib.request, warnings

warnings.filterwarnings("ignore")

# --- 1. CONFIGURATION FLAGS ---
USE_SN             = True
USE_DESI_BAO       = True
USE_TRANSVERSE_BAO = False
USE_FS8            = True
USE_CMB            = False
USE_CC             = True

suffix = "THESIS_FULL_SET"
N_data = 0
if USE_SN:         suffix += "_SN";    N_data += 1590
if USE_DESI_BAO:   suffix += "_DESI";  N_data += 14
if USE_TRANSVERSE_BAO: suffix += "_TRBAO"
if USE_FS8:        suffix += "_FS8";   N_data += 7
if USE_CMB:        suffix += "_CMB";   N_data += 3
if USE_CC:         suffix += "_CC";    N_data += 34

print(f"Analysis Scenario: {suffix} | Total Data Points (N_data): {N_data}")

# --- 2. LOAD DATASETS ---
if USE_SN:
    url_dat = "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/Pantheon%2BSH0ES.dat"
    url_cov = "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/Pantheon%2BSH0ES_STAT%2BSYS.cov"
    if not os.path.exists("Pantheon+SH0ES.dat"): urllib.request.urlretrieve(url_dat, "Pantheon+SH0ES.dat")
    if not os.path.exists("Pantheon+SH0ES.cov"): urllib.request.urlretrieve(url_cov, "Pantheon+SH0ES.cov")
    df_sn = pd.read_csv("Pantheon+SH0ES.dat", sep=r'\s+')
    mask = df_sn['zHD'] > 0.01
    z_sn, mu_obs = df_sn[mask]['zHD'].values, df_sn[mask]['m_b_corr'].values
    with open("Pantheon+SH0ES.cov", 'r') as f:
        n_total = int(f.readline())
        cov_flat = np.loadtxt(f)
    inv_C_sn = np.linalg.inv(cov_flat.reshape((n_total, n_total))[np.ix_(np.where(mask.values)[0], np.where(mask.values)[0])])

if USE_DESI_BAO or USE_FS8:
    z_obs       = np.array([0.15, 0.51, 0.71, 0.93, 1.11, 1.32, 1.49])
    dm_rs_obs   = np.array([4.47, 13.62, 16.85, 21.71, 25.46, 27.81, 31.06])
    dh_rs_obs   = np.array([28.47, 20.98, 20.07, 16.09, 13.91, 13.14, 11.23])
    dm_rs_err   = np.array([0.15, 0.20, 0.22, 0.25, 0.30, 0.35, 0.45])
    dh_rs_err   = np.array([0.40, 0.45, 0.45, 0.50, 0.60, 0.70, 0.90])
    desi_r_corr = np.array([-0.40, -0.40, -0.40, -0.40, -0.40, -0.40, -0.40])
    fs8_obs     = np.array([0.444, 0.462, 0.451, 0.475, 0.482, 0.467, 0.432])
    fs8_err     = np.array([0.038, 0.040, 0.031, 0.025, 0.030, 0.045, 0.050])

if USE_TRANSVERSE_BAO:
    bao_tr_z   = np.array([0.38, 0.51, 0.61, 0.81, 1.52, 2.33])
    bao_tr_obs = np.array([10.23, 13.36, 15.45, 18.92, 26.69, 37.77])
    bao_tr_err = np.array([0.17, 0.21, 0.22, 0.51, 0.90, 2.13])
    N_data += len(bao_tr_z)

cc_z = np.array([0.07, 0.09, 0.12, 0.17, 0.179, 0.199, 0.2, 0.27, 0.28, 0.35, 0.352, 0.38, 0.4, 0.4004, 0.424, 0.44, 0.47, 0.4783, 0.48, 0.57, 0.593, 0.68, 0.73, 0.781, 0.875, 0.88, 0.9, 1.037, 1.3, 1.363, 1.43, 1.53, 1.75, 1.965])
cc_h = np.array([69.0, 69.0, 68.5, 83.0, 75.0, 75.0, 72.9, 77.0, 88.8, 82.7, 83.0, 83.0, 95.0, 77.0, 87.6, 82.6, 89.0, 80.9, 97.0, 92.4, 104.0, 92.0, 97.3, 105.0, 125.0, 90.0, 117.0, 154.0, 168.0, 160.0, 177.0, 140.0, 202.0, 186.5])
cc_err = np.array([19.6, 12.0, 15.0, 8.0, 4.0, 5.0, 29.6, 14.0, 36.6, 8.4, 14.0, 13.5, 17.0, 10.2, 7.8, 7.8, 49.6, 9.0, 62.0, 4.5, 13.0, 8.0, 7.0, 12.0, 17.0, 40.0, 23.0, 20.0, 17.0, 33.6, 18.0, 14.0, 40.0, 50.4])

z_grid = np.linspace(0, 1500, 2500)

def get_model_setup(mod):
    # Model parameters initialization and standard deviations
    if mod == 'LCDM':
        labels, p0, p_std = [r"$\Omega_m$", r"$h$", r"$\omega_b$"], [0.31, 0.68, 0.0225], [0.01, 0.01, 0.0001]
    elif mod == 'wCDM':
        labels, p0, p_std = [r"$\Omega_m$", r"$w_0$", r"$h$", r"$\omega_b$"], [0.31, -1.0, 0.68, 0.0225], [0.01, 0.05, 0.01, 0.0001]
    elif mod == 'CPL':
        labels, p0, p_std = [r"$\Omega_m$", r"$w_0$", r"$w_a$", r"$h$", r"$\omega_b$"], [0.31, -1.0, 0.0, 0.68, 0.0225], [0.01, 0.05, 0.1, 0.01, 0.0001]

    if USE_FS8:
        labels.append(r"$\sigma_8$")
        p0.append(0.8)
        p_std.append(0.02)

    return len(p0), p0, p_std, labels

def E_z(z, Om, w0, wa, h):
    Or0  = 4.15e-5 / (h**2)
    f_de = ((1.0 + z)**(3.0*(1.0 + w0 + wa))) * np.exp(-3.0*wa*(z/(1.0 + z)))
    return np.sqrt(Or0*(1.0+z)**4.0 + Om*(1.0+z)**3.0 + (1.0-Om-Or0)*f_de)

v_mean_cmb = np.array([1.7428, 301.406, 0.02259])
cov_cmb = np.array([
    [ 2.8090e-05,  2.1465e-04, -6.2163e-07],
    [ 2.1465e-04,  8.1000e-03, -5.2020e-06],
    [-6.2163e-07, -5.2020e-06,  2.8900e-08]
])
inv_cov_cmb = np.linalg.inv(cov_cmb)

def get_fs8_array(z_array, Om, w0, wa, h, s8_norm):
    a_ini = 1.0 / (1.0 + 100.0)

    def odesys(a, y):
        delta, ddelta_da = y
        Or0 = 4.15e-5 / (h**2)
        Ode0 = 1.0 - Om - Or0
        f_de = (a**(-3.0*(1.0 + w0 + wa))) * np.exp(-3.0 * wa * (1.0 - a))
        E2 = Or0 * a**(-4) + Om * a**(-3) + Ode0 * f_de
        dfde_da = f_de * (-3.0 * (1.0 + w0 + wa) / a + 3.0 * wa)
        dE2_da = -4.0 * Or0 * a**(-5) - 3.0 * Om * a**(-4) + Ode0 * dfde_da
        dlnE_da = dE2_da / (2.0 * E2)
        d2delta_da2 = - (3.0 / a + dlnE_da) * ddelta_da + (3.0 * Om / (2.0 * a**5 * E2)) * delta
        return [ddelta_da, d2delta_da2]

    sol = solve_ivp(odesys, [a_ini, 1.0], [a_ini, 1.0], dense_output=True, method='RK45', rtol=1e-5, atol=1e-8)
    delta_0 = sol.sol(1.0)[0]

    a_array = 1.0 / (1.0 + z_array)
    y_obs = sol.sol(a_array)
    f_obs = (a_array / y_obs[0]) * y_obs[1]

    return f_obs * s8_norm * (y_obs[0] / delta_0)

def log_likelihood(theta, model):
    if model == 'LCDM':
        Om, h, wb = theta[0], theta[1], theta[2]
        w0, wa = -1.0, 0.0
        idx = 3
    elif model == 'wCDM':
        Om, w0, h, wb = theta[0], theta[1], theta[2], theta[3]
        wa = 0.0
        idx = 4
    elif model == 'CPL':
        Om, w0, wa, h, wb = theta[0], theta[1], theta[2], theta[3], theta[4]
        idx = 5

    if USE_FS8:
        s8 = theta[idx]
    else:
        s8 = 0.8

    if not (0.1 < Om < 0.6 and 0.5 < h < 0.9):       return -np.inf
    if not (0.01 < wb < 0.04):                       return -np.inf
    if model != 'LCDM' and not (-3.0 < w0 < 1.0):    return -np.inf
    if model == 'CPL'  and not (-4.0 < wa <  4.0):   return -np.inf
    if USE_FS8 and not (0.2 < s8 < 1.5):             return -np.inf

    chi2_val = 0.0

    if not USE_CMB:
        chi2_val += ((wb - 0.02218) / 0.00055)**2

    wm = Om * h**2
    rd_theo = 55.154 * np.exp(-72.3 * (wb - 0.02218)**2) / (wm**0.25351 * wb**0.12807)

    f_int = interp1d(
        z_grid,
        cumulative_trapezoid(1.0 / E_z(z_grid, Om, w0, wa, h), z_grid, initial=0),
        kind='linear', bounds_error=False, fill_value="extrapolate"
    )

    if USE_CMB:
        g1 = 0.0783 * wb**(-0.238) / (1.0 + 39.5 * wb**0.763)
        g2 = 0.560 / (1.0 + 21.1 * wb**1.81)
        z_star = 1048.0 * (1.0 + 0.00124 * wb**(-0.738)) * (1.0 + g1 * wm**g2)
        a_star = 1.0 / (1.0 + z_star)
        a_rs = np.linspace(1e-8, a_star, 1000)
        z_rs_array = 1.0 / a_rs - 1.0
        R_b = 31500.0 * wb * (2.7255 / 2.7)**(-4)
        cs_a = 299792.458 / np.sqrt(3.0 * (1.0 + R_b * a_rs))
        Hz_a = 100.0 * h * E_z(z_rs_array, Om, w0, wa, h)
        rs_star = np.trapz(cs_a / (a_rs**2 * Hz_a), a_rs)
        R_calc = np.sqrt(Om) * f_int(z_star)
        DM_star = (299792.458 / (100.0 * h)) * f_int(z_star)
        la_calc = np.pi * DM_star / rs_star
        diff_cmb = np.array([R_calc, la_calc, wb]) - v_mean_cmb
        chi2_val += diff_cmb.T @ inv_cov_cmb @ diff_cmb

    if USE_SN:
        mu_theo_0 = 5.0 * np.log10((299792.458 / (100.0 * h)) * (1.0 + z_sn) * f_int(z_sn)) + 25.0
        delta_mu = mu_obs - mu_theo_0
        chi2_val += (delta_mu.T @ inv_C_sn @ delta_mu) - (np.sum(inv_C_sn @ delta_mu)**2 / np.sum(inv_C_sn))

    if USE_DESI_BAO or USE_FS8:
        if USE_FS8:
            fs8_theo_array = get_fs8_array(z_obs, Om, w0, wa, h, s8)

        for i, zi in enumerate(z_obs):
            Ez_i = E_z(zi, Om, w0, wa, h)
            if USE_DESI_BAO:
                dm = (299792.458/(100.0*h)) * f_int(zi) / rd_theo
                dh = (299792.458/(100.0*h*Ez_i)) / rd_theo
                diff_bao = np.array([dm - dm_rs_obs[i], dh - dh_rs_obs[i]])
                cov_bao  = np.array([
                    [dm_rs_err[i]**2, desi_r_corr[i]*dm_rs_err[i]*dh_rs_err[i]],
                    [desi_r_corr[i]*dm_rs_err[i]*dh_rs_err[i], dh_rs_err[i]**2]
                ])
                chi2_val += diff_bao.T @ np.linalg.inv(cov_bao) @ diff_bao
            if USE_FS8:
                chi2_val += ((fs8_theo_array[i] - fs8_obs[i]) / fs8_err[i])**2

    if USE_TRANSVERSE_BAO:
        for i, zi in enumerate(bao_tr_z):
            dm_theo = (299792.458/(100.0*h)) * f_int(zi) / rd_theo
            chi2_val += ((dm_theo - bao_tr_obs[i]) / bao_tr_err[i])**2

    if USE_CC:
        chi2_val += np.sum(((100.0*h*E_z(cc_z, Om, w0, wa, h) - cc_h) / cc_err)**2)

    return -0.5 * chi2_val

# --- 3. RUN MCMC ---
n_steps, burn_in = 15000, 3000
N_walkers = 32

for mod in ['LCDM', 'wCDM', 'CPL']:
    ndim, p0, p_std, _ = get_model_setup(mod)
    print(f"\nRunning MCMC for {mod} (ndim={ndim}, walkers={N_walkers}, steps={n_steps})...")
    filename = f"mcmc_chain_{mod}_{suffix}.h5"
    if os.path.exists(filename): os.remove(filename)

    backend  = emcee.backends.HDFBackend(filename)
    backend.reset(N_walkers, ndim)
    sampler  = emcee.EnsembleSampler(N_walkers, ndim, log_likelihood, args=[mod], backend=backend)

    # Initialize walkers
    pos_ini = np.array(p0) + np.array(p_std) * np.random.randn(N_walkers, ndim)

    sampler.run_mcmc(pos_ini, n_steps, progress=True)
    print(f"    Acceptance rate: {np.mean(sampler.acceptance_fraction)*100:.1f}%")

def gelman_rubin(chain):
    N, M     = chain.shape[0], chain.shape[1]
    mean_w   = np.mean(chain, axis=0)
    mean_all = np.mean(mean_w, axis=0)
    B = (N/(M-1)) * np.sum((mean_w - mean_all)**2, axis=0)
    W = (1/(M*(N-1))) * np.sum((chain - mean_w)**2, axis=(0,1))
    return np.sqrt(((N-1)/N)*W + (1/N)*B) / np.sqrt(W)

def analyze_results():
    models_internal = ['LCDM', 'wCDM', 'CPL']
    display_names   = {'LCDM': r'$\Lambda$CDM', 'wCDM': r'$w$CDM', 'CPL': 'CPL'}
    colors          = {'LCDM': '#2ca02c',        'wCDM': '#1f77b4', 'CPL': '#d62728'}

    print("\n" + "="*95)
    print(f"FINAL ANALYSIS: {suffix}")
    print("="*95)

    chains, results_list = {}, []

    for mod in models_internal:
        _, _, _, labels = get_model_setup(mod)
        filename = f"mcmc_chain_{mod}_{suffix}.h5"
        reader   = emcee.backends.HDFBackend(filename, read_only=True)
        flat     = reader.get_chain(discard=burn_in, flat=True)
        raw      = reader.get_chain(discard=burn_in)
        chains[mod] = flat

        meds         = np.median(flat, axis=0)
        sig_up, sig_lo = np.percentile(flat, 84, axis=0) - meds, meds - np.percentile(flat, 16, axis=0)
        chi2_min       = -2 * log_likelihood(meds, mod)
        r_hat          = np.max(gelman_rubin(raw))

        results_list.append({'mod': mod, 'pretty_mod': display_names[mod], 'meds': meds})

        print(f"\nModel: {display_names[mod]}")
        convergence_status = "Converged" if r_hat < 1.05 else "Non-converged"
        print(f"    R-hat (max): {r_hat:.4f} ({convergence_status})   |   χ² min: {chi2_min:.2f}")
        for i in range(len(meds)):
            print(f"    {labels[i]:>12s}: {meds[i]:.4f}  +{sig_up[i]:.4f}  -{sig_lo[i]:.4f}")

        if mod == 'CPL':
            cov_w0_wa      = np.cov(flat[:, 1], flat[:, 2])
            diff           = np.array([-1.0 - np.mean(flat[:, 1]), 0.0 - np.mean(flat[:, 2])])
            delta_chi2_2d  = diff.T @ np.linalg.inv(cov_w0_wa) @ diff
            sigma_2d       = norm.isf(chi2.sf(delta_chi2_2d, df=2) / 2)
            print(f"    Deviation from (-1,0): Δχ² = {delta_chi2_2d:.2f}  (~{sigma_2d:.2f}σ)")

        # Generate corner plots
        fig = corner.corner(
            flat, labels=labels, color=colors[mod],
            show_titles=True, title_fmt=".4f", quantiles=[0.16, 0.5, 0.84],
            levels=(0.68, 0.95), smooth=1.1, bins=40,
            fill_contours=True, plot_datapoints=False, alpha=0.6
        )
        fig.subplots_adjust(top=0.88)
        fig.suptitle(f"{display_names[mod]} Constraints", fontsize=14, y=0.98)
        plt.show()

    print("\n── Model Comparison ──────────────────────")
    print(f"{'Model':<10} {'k':>4} {'χ²_min':>10} {'AIC':>10} {'BIC':>10}")
    for r in results_list:
        k_base = {'LCDM': 4, 'wCDM': 5, 'CPL': 6}[r['mod']]
        chi2_min = -2 * log_likelihood(r['meds'], r['mod'])
        aic      = chi2_min + 2 * k_base
        bic      = chi2_min + k_base * np.log(N_data)
        print(f"{r['pretty_mod']:<10} {k_base:>4} {chi2_min:>10.2f} {aic:>10.2f} {bic:>10.2f}")

analyze_results()